In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pyxtal import pyxtal

In [3]:
from abtem.visualize.visualizations import show_atoms

In [187]:
import numpy as np
from ase import Atoms
from skimage.filters import gaussian
from pyxtal import pyxtal

# code adapted from: https://github.com/jarek-pawlowski/wallpaper_group/blob/main/utils_gen.py

plane_to_layer_groups = {
    1:  [1, 4, 5],
    2:  [2, 3, 6, 7],
    3:  [8, 11, 27, 28, 36],
    4:  [9, 12, 29, 32, 33],
    5:  [10, 13, 34, 35],
    6:  [14, 19, 23, 37, 38, 41, 48],
    7:  [15, 16, 20, 24, 40, 43, 45],
    8:  [17, 21, 25, 44],
    9:  [18, 22, 26, 42, 47],
    10: [39, 46, 49, 50, 51],
    11: [53, 55, 57, 59, 61, 62, 64],
    12: [52, 54, 56, 58, 60, 63],
    13: [65, 66, 74],
    14: [67, 69],
    15: [68, 70],
    16: [66, 73, 75],
    17: [71, 72, 76, 77]
}

# mapping in original GitHub repo is wrong.
# "name" = wallpaper group name,
# "layer_num" = layer group number,
# "num_atoms" = bounds for number of atoms in unit cell,
# "even" = shoud number of atoms in unit cell be even? some layer group only accept even number atoms
wallpaper_groups = {1:  {"name" : "p1",   "layer_num" : 4,  "num_atoms" : [1, 10], "even" : False},  # 1, 4, 5
                    2:  {"name" : "p2",   "layer_num" : 3,  "num_atoms" : [3, 10], "even" : False},  # 2, 3, 6, 7
                    3:  {"name" : "pm",   "layer_num" : 27, "num_atoms" : [3, 10], "even" : True},   # 8, 11, 27, 28, 36
                    4:  {"name" : "pg",   "layer_num" : 29, "num_atoms" : [4, 10], "even" : True},   # 9, 12, 29, 32, 33
                    5:  {"name" : "pmm",  "layer_num" : 35, "num_atoms" : [4, 16], "even" : True},   # 10, 13, 34, 35
                    6:  {"name" : "pmg",  "layer_num" : 23, "num_atoms" : [4, 16], "even" : True},   # 14, 19, 23, 37, 38, 41, 48
                    7:  {"name" : "pgg",  "layer_num" : 24, "num_atoms" : [4, 16], "even" : True},   # 15, 16, 20, 24, 40, 43, 45
                    8:  {"name" : "cm",   "layer_num" : 21, "num_atoms" : [4, 16], "even" : True},   # 17, 21, 25, 44
                    9:  {"name" : "cmm",  "layer_num" : 26, "num_atoms" : [4, 16], "even" : True},   # 18, 22, 26, 42, 47
                    10: {"name" : "p4",   "layer_num" : 49, "num_atoms" : [4, 20], "even" : True},   # 39, 46, 49, 50, 51
                    11: {"name" : "p4m",  "layer_num" : 55, "num_atoms" : [4, 20], "even" : True},   # 53, 55, 57, 59, 61, 62, 64
                    12: {"name" : "p4g",  "layer_num" : 56, "num_atoms" : [4, 20], "even" : True},   # 52, 54, 56, 58, 60, 63
                    13: {"name" : "p3",   "layer_num" : 65, "num_atoms" : [4, 20], "even" : True},   # 65, 66, 74
                    14: {"name" : "p3m1", "layer_num" : 69, "num_atoms" : [4, 20], "even" : True},   # 67, 69
                    15: {"name" : "p31m", "layer_num" : 70, "num_atoms" : [4, 20], "even" : True},   # 68, 70
                    16: {"name" : "p6",   "layer_num" : 73, "num_atoms" : [6, 20], "even" : True},   # 66, 73, 75
                    17: {"name" : "p6m",  "layer_num" : 77, "num_atoms" : [6, 20], "even" : True}}   # 71, 72, 76, 77

def make_cell_rectangular(atoms: Atoms):
    new_atoms = atoms.copy()
    cell = new_atoms.cell.copy()

    # in xy plane
    cell[0, 1] = 0  # Remove xy shear
    cell[1, 0] = 0  # Remove yx shear

    new_atoms.set_cell(cell, scale_atoms=False) # scale_atoms=False, Prevent atomic position scaling
    new_atoms.wrap()  # Ensure atoms are inside the new cell
    return new_atoms

def make_cell_square(atoms: Atoms):
    new_atoms = make_cell_rectangular(atoms)
    cell = new_atoms.cell.copy()

    # Create the new square cell
    size = min(cell[0, 0], cell[1, 1])
    cell[0, 0] = size
    cell[1, 1] = size

    # Before set_cell, we have to make periodic boundary condition False,
    # then scaled_positions won't be scaled to [0, 1]
    new_atoms.set_pbc(False)

    # Apply the new cell
    new_atoms.set_cell(cell, scale_atoms=False)  # positions do not change, but scaled_positions updated
    scaled_positions = new_atoms.get_scaled_positions()
    # Remove atoms that are outside [0,1) in any direction
    mask = (scaled_positions >= 0).all(axis=1) & (scaled_positions < 1).all(axis=1)
    new_atoms_ = new_atoms[mask]  # Remove atoms outside the new cell

    return new_atoms_

def rotate_atoms_xy_center(atoms, angle_deg):
    """
    Rotate an ASE Atoms object in the XY plane around the center (a/2, b/2).

    Parameters
    ----------
    atoms : ase.Atoms
        The Atoms object to be rotated. Assumes orthogonal cell with a = b.
    angle_deg : float
        The rotation angle in degrees (counterclockwise).
    """
    a = atoms.cell[0, 0]
    b = atoms.cell[1, 1]
    center = (a / 2, b / 2, 0)
    atoms.rotate('z', angle_deg, center=center, rotate_cell=False)
    return atoms

import numpy as np
from ase import Atoms

import numpy as np
from ase import Atoms

import numpy as np
from ase import Atoms

def crop_atoms_xy_center(atoms, a_new=None, b_new=None):
    """
    Crop atoms from the center of the cell in the XY plane and update the cell.

    Parameters
    ----------
    atoms : ase.Atoms
        The original Atoms object.
    a_new : float, optional
        New width in x-direction. Default is a / sqrt(2).
    b_new : float, optional
        New height in y-direction. Default is b / sqrt(2).

    Returns
    -------
    ase.Atoms
        A new Atoms object cropped from the center, with updated cell.
    """
    a = atoms.cell[0, 0]
    b = atoms.cell[1, 1]
    c = atoms.cell[2, 2]

    if a_new is None:
        a_new = a / np.sqrt(2)
    if b_new is None:
        b_new = b / np.sqrt(2)

    center = np.array([a / 2, b / 2])
    lower = center - np.array([a_new / 2, b_new / 2])
    upper = center + np.array([a_new / 2, b_new / 2])

    # Filter atoms in the new region
    positions = atoms.get_positions()
    in_crop = ((positions[:, 0] >= lower[0]) & (positions[:, 0] <= upper[0]) &
               (positions[:, 1] >= lower[1]) & (positions[:, 1] <= upper[1]))

    cropped = atoms[in_crop].copy()

    # Shift positions so new cell starts at (0, 0, 0)
    cropped.positions -= np.array([lower[0], lower[1], 0])

    # Update cell
    new_cell = atoms.cell.copy()
    new_cell[0, 0] = a_new
    new_cell[1, 1] = b_new
    cropped.set_cell(new_cell)
    cropped.set_pbc(atoms.get_pbc())

    return cropped



class RandomLattice:

    def __init__(self, supercell=[15,15,1], max_rotation_angle=90, atom_radius=.3, scale=1.):

        self.max_rotation_angle = max_rotation_angle
        self.supercell = supercell
        self.atom_radius = atom_radius
        self.scale = scale
        self.group = None
        self.atoms = None
        self.atoms_unit_cell = None
        self.atoms_ase = None

    def generate_lattice(self, wallpaper_class, seed=None):
        # Create a SeedSequence from the main seed
        seed_seq = np.random.SeedSequence(seed)

        # Generate independent child seeds
        child_seeds = seed_seq.spawn(3)

        # Create independent random generators
        rng1 = np.random.default_rng(child_seeds[0]) # this is for num_atoms_in_unit_cell
        rng2 = np.random.default_rng(child_seeds[1]) # this is for construct random structure
        rng3 = np.random.default_rng(child_seeds[2]) # this is for rotation?

        if wallpaper_class not in range(1, 18):
            raise ValueError("wallpaper_class should be between 1 and 17")
        self.group = wallpaper_groups[wallpaper_class]
        # randomize no of atoms in unit cell
        min_num_atoms_in_unit_cell, max_num_atoms_in_unit_cell = self.group["num_atoms"]
        if self.group["even"]:
            rand_bounds = [int(min_num_atoms_in_unit_cell/2), int(max_num_atoms_in_unit_cell/2)]
            rand_multiplier = 2
        else:
            rand_bounds = [min_num_atoms_in_unit_cell, max_num_atoms_in_unit_cell]
            rand_multiplier = 1
        num_atoms_in_unit_cell = rng1.integers(rand_bounds[0], rand_bounds[1], endpoint=True) * rand_multiplier
        # define random crystal
        struct = pyxtal()
        _ = struct.from_random(dim=2, group=self.group["layer_num"],
                               species=['C'], numIons=[num_atoms_in_unit_cell],
                               thickness=2., random_state=rng2)
        self.atoms_unit_cell = struct.to_ase()
        self.ase_atoms = self.atoms_unit_cell * self.supercell
        self.atoms = make_cell_square(self.ase_atoms)
        # get a random angle
        angle_deg = rng3.uniform(0, 360)
        self.atoms = rotate_atoms_xy_center(self.atoms, angle_deg)
        self.atoms = crop_atoms_xy_center(self.atoms)
        # ase_lattice.rotate(np.random.randint(self.max_rotation_angle)+1, 'z')
        return self.atoms

    def generate_data(self, ase_atoms, size=512):
        a = ase_atoms.cell.cellpar()[0]
        xyz = ase_atoms.get_positions() * (size - 1) / a
        x = np.round(xyz[:, 0]).astype(int)
        y = np.round(xyz[:, 1]).astype(int)
        shape = (size, size)
        array = np.zeros(shape)
        array[x, y] = 1.
        array = gaussian(array, sigma=5, mode='constant')
        return array

In [199]:
aa = RandomLattice()
lattice = aa.generate_lattice(13, seed=48)
kk = aa.generate_data(lattice)

In [200]:
plt.imshow(kk)

In [183]:
show_atoms(aa.atoms_unit_cell)

(<Figure size 960x720 with 1 Axes>, <Axes: xlabel='x [Å]', ylabel='y [Å]'>)

In [184]:
show_atoms(aa.ase_atoms)

(<Figure size 960x720 with 1 Axes>, <Axes: xlabel='x [Å]', ylabel='y [Å]'>)

In [185]:
show_atoms(aa.atoms, plane='xy')

(<Figure size 960x720 with 1 Axes>, <Axes: xlabel='x [Å]', ylabel='y [Å]'>)

In [190]:
plt.imshow(kk)

## what random_state controls

* unit cell parameters
* atom positions